# Skill Smoke Test — validate all SKILL.md files

Structural checks from the `skill-smoke-test` skill's table. Run all cells, read the
PASS/FAIL report at the end, fix FAIL items, re-run, then commit.

This validates **structure**, not model dispatch — a perfect skill can still fail to load
if the model routes elsewhere. For full verification, do the manual chat test from the
skill's procedure (step 4).

## Load every skill and parse its frontmatter

Checks: YAML frontmatter parses cleanly (an unescaped colon or tab indent breaks the
loader silently), `name` matches the folder name (mismatch = invisible skill), and
`description` exists.

In [1]:
import os, re, yaml

SKILLS_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
# notebook lives at .github/skills/skill-smoke-test/notebooks/ -> repo root is 3 levels up
if not os.path.isdir(os.path.join(SKILLS_DIR, '.github', 'skills')):
    SKILLS_DIR = os.path.abspath('.')  # running from repo root (nbdev_test)
SKILLS = {}

for folder in sorted(os.listdir(os.path.join(SKILLS_DIR, '.github', 'skills'))):
    path = os.path.join(SKILLS_DIR, '.github', 'skills', folder, 'SKILL.md')
    if not os.path.isfile(path):
        continue
    text = open(path, encoding='utf-8').read()
    fm_match = re.match(r'^---\n(.*?)\n---', text, re.S)
    if not fm_match:
        SKILLS[folder] = {'path': path, 'text': text, 'fm': None, 'fm_error': 'no frontmatter block'}
        continue
    try:
        fm = yaml.safe_load(fm_match.group(1))
        SKILLS[folder] = {'path': path, 'text': text, 'fm': fm or {}, 'fm_error': None}
    except yaml.YAMLError as e:
        SKILLS[folder] = {'path': path, 'text': text, 'fm': None, 'fm_error': str(e)}

print(f'Loaded {len(SKILLS)} skills from .github/skills/')
print('Frontmatter parse errors:', [k for k, v in SKILLS.items() if v['fm_error']] or 'none')

Loaded 41 skills from .github/skills/
Frontmatter parse errors: none


## Structural checks per skill

- `name` matches folder; only letters/numbers/hyphens.
- `description` non-empty and keyword-rich (>= 80 chars).
- `user_invocable` / `disable_model_invocation`, when present, are booleans and not both
  `false`-equivalent in a way that makes the skill unreachable.
- Relative references (`./scripts/`, `./assets/`, `./notebooks/`) exist on disk.
- Honesty: a skill whose description contains `PLACEHOLDER` must also say `NOT implemented`
  (keeps the library's honesty contract consistent).

In [2]:
REPORT = []

for folder, s in SKILLS.items():
    fails = []
    fm, text = s['fm'], s['text']
    if fm is None:
        fails.append(f'frontmatter unparseable: {s["fm_error"]}')
    else:
        name = fm.get('name')
        if name != folder:
            fails.append(f'name "{name}" != folder "{folder}"')
        if name and not re.fullmatch(r'[A-Za-z0-9-]+', str(name)):
            fails.append(f'name has invalid characters: {name}')
        desc = str(fm.get('description') or '')
        if len(desc) < 80:
            fails.append(f'description missing or too short ({len(desc)} chars)')
        ui, dmi = fm.get('user_invocable'), fm.get('disable_model_invocation')
        for flag, val in (('user_invocable', ui), ('disable_model_invocation', dmi)):
            if val is not None and not isinstance(val, bool):
                fails.append(f'{flag} is not a bool: {val!r}')
        for rel in re.findall(r'\./(?:scripts|assets|notebooks)/[\w.-]+', text):
            p = os.path.join(SKILLS_DIR, '.github', 'skills', folder, rel)
            if not os.path.exists(p):
                fails.append(f'broken relative reference: {rel}')
        if 'PLACEHOLDER' in desc and 'NOT implemented' not in desc:
            fails.append('placeholder skill does not say "NOT implemented"')
    REPORT.append({'skill': folder, 'ok': not fails, 'fails': fails})

n_bad = sum(not r['ok'] for r in REPORT)
print(f'Checked {len(REPORT)} skills — {n_bad} FAIL, {len(REPORT) - n_bad} PASS\n')
for r in REPORT:
    status = 'PASS' if r['ok'] else 'FAIL'
    print(f'[{status}] {r["skill"]}')
    for f in r['fails']:
        print(f'        - {f}')

Checked 41 skills — 0 FAIL, 41 PASS

[PASS] active-learning
[PASS] choose-augmentation
[PASS] choose-channels
[PASS] choose-loss-function
[PASS] choose-model-architecture
[PASS] choose-tiling-strategy
[PASS] classical-segmentation-baseline
[PASS] configure-metrics-checkpointing
[PASS] configure-training-schedule
[PASS] continuous-integration
[PASS] custom-callbacks-howto
[PASS] data-versioning
[PASS] dataset-imbalance-strategies
[PASS] distributed-training
[PASS] experiment-tracking
[PASS] export-onnx
[PASS] fast-network-data-copy
[PASS] feature-channels
[PASS] git-last-change-summary
[PASS] learning-rate-finder
[PASS] lifted-chip-mask-synthesis
[PASS] mask-preprocessing
[PASS] mixed-precision-deep-dive
[PASS] model-comparison-ab-test
[PASS] model-ensembling
[PASS] model-interpretability
[PASS] multi-class-segmentation
[PASS] nbdev-export
[PASS] nbdev-notebook-verify
[PASS] nbdev-uv-new-project
[PASS] object-level-metrics
[PASS] production-deployment
[PASS] pseudo-labeling
[PASS] run-i

## Overlap report (informational)

The skill's "extend, don't split" rule: two skills covering the same topic drift out of
sync. This coarse check flags skills whose descriptions share several rare keywords —
manual judgement required, not an automatic FAIL.

In [3]:
from collections import defaultdict

STOP = {'use', 'when', 'user', 'asks', 'asking', 'want', 'wants', 'edit', 'editing',
        'block', 'emit', 'emits', 'set', 'setting', 'get', 'not', 'yet', 'implemented'}

def keywords(desc):
    words = re.findall(r'[a-z][a-z-]{3,}', desc.lower())
    return {w for w in words if w not in STOP}

kw_to_skills = defaultdict(list)
for folder, s in SKILLS.items():
    if s['fm'] and s['fm'].get('description'):
        for kw in keywords(str(s['fm']['description'])):
            kw_to_skills[kw].append(folder)

overlap = {kw: sk for kw, sk in kw_to_skills.items() if len(sk) >= 3}
print('Keywords shared by >= 3 skills (review manually, not automatic FAILs):')
for kw, sk in sorted(overlap.items(), key=lambda kv: -len(kv[1])):
    print(f'  {kw}: {", ".join(sk)}')
if not overlap:
    print('  none')

Keywords shared by >= 3 skills (review manually, not automatic FAILs):
  segmentation: active-learning, choose-augmentation, choose-channels, choose-loss-function, choose-model-architecture, choose-tiling-strategy, classical-segmentation-baseline, configure-metrics-checkpointing, configure-training-schedule, continuous-integration, custom-callbacks-howto, data-versioning, dataset-imbalance-strategies, distributed-training, experiment-tracking, export-onnx, feature-channels, learning-rate-finder, mask-preprocessing, mixed-precision-deep-dive, model-comparison-ab-test, model-ensembling, model-interpretability, multi-class-segmentation, object-level-metrics, production-deployment, pseudo-labeling, run-inference, security-data-privacy, segmentation-eda-validation, segmentation-error-analysis, segmentation-project-planner, test-time-augmentation, training-monitoring-callbacks
  tools: active-learning, choose-augmentation, choose-channels, choose-loss-function, choose-model-architecture, cho

## Manual test prompts

For each PASS skill, paste one of these into the chat to confirm it actually loads and
follows its procedure (structure ≠ dispatch):

In [4]:
for r in REPORT:
    if r['ok']:
        print(f'  /{r["skill"]}  — or: "Invoke the {r["skill"]} skill and follow its procedure for a toy example."')

  /active-learning  — or: "Invoke the active-learning skill and follow its procedure for a toy example."
  /choose-augmentation  — or: "Invoke the choose-augmentation skill and follow its procedure for a toy example."
  /choose-channels  — or: "Invoke the choose-channels skill and follow its procedure for a toy example."
  /choose-loss-function  — or: "Invoke the choose-loss-function skill and follow its procedure for a toy example."
  /choose-model-architecture  — or: "Invoke the choose-model-architecture skill and follow its procedure for a toy example."
  /choose-tiling-strategy  — or: "Invoke the choose-tiling-strategy skill and follow its procedure for a toy example."
  /classical-segmentation-baseline  — or: "Invoke the classical-segmentation-baseline skill and follow its procedure for a toy example."
  /configure-metrics-checkpointing  — or: "Invoke the configure-metrics-checkpointing skill and follow its procedure for a toy example."
  /configure-training-schedule  — or: "Invok

## Summary

FAIL items above must be fixed and the notebook re-run before committing a new or edited
skill. See `.github/skills/skill-smoke-test/SKILL.md` and
`tutorials/03_testing_skills.ipynb` for the full workflow.